In [1]:
#import
import ConnectionConfig as cc
debugging_mode=True

In [5]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("AnalyseVragenS1",4)
spark.getActiveSession()

Environment variables are set...


In [8]:
#Dimensies en fact inladen
dateDimDf= spark.read.format("delta").load("./delta/DATE_DIM")
rainDimDf= spark.read.format("delta").load("./delta/RAIN_DIM")
seasonDimDf= spark.read.format("delta").load("./delta/SEASON_DIM")
userDimDf = spark.read.format("delta").load("./spark-warehouse/dimuser")
treasureTypeDimDf = spark.read.format("delta").load("./spark-warehouse/dimtreasuretype")
treasureFoundFactDf = spark.read.format("delta").load("./delta/FACT_TREASURE_FOUND")

dateDimDf.createOrReplaceTempView("dimDate")
rainDimDf.createOrReplaceTempView("dimRain")
seasonDimDf.createOrReplaceTempView("dimSeason")
userDimDf.createOrReplaceTempView("dimUser")
treasureTypeDimDf.createOrReplaceTempView("dimTreasureType")
treasureFoundFactDf.createOrReplaceTempView("factTreasureFound")

In [4]:
print("=== VRAAG 1: Datumparameters effect op Treasurehunts ===")

# Analyse op verschillende datumniveaus
query1 = """
SELECT
    d.Year,
    d.Month,
    d.Week,
    COUNT(*) as AantalTreasureHunts,
    ROUND(AVG(COUNT(*)) OVER (PARTITION BY d.Year), 2) as GemiddeldePerJaar,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as PercentageTotaal
FROM factTreasureFound f
JOIN dimDate d ON f.DateSurKey = d.DateSurKey
GROUP BY d.Year, d.Month, d.Week
ORDER BY d.Year, d.Month, d.Week
"""

result1 = spark.sql(query1)
print("Jaar/Maand/Week analyse:")
result1.show(20)

# Seizoenen analyse
query1_season = """
SELECT
    s.SeasonName as Seizoen,
    COUNT(*) as AantalTreasureHunts,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM factTreasureFound), 2) as Percentage,
    ROUND(AVG(f.duration), 2) as GemiddeldeZoekTijd
FROM factTreasureFound f
JOIN dimSeason s ON f.SeasonSurKey = s.SeasonSurKey
GROUP BY s.SeasonName
ORDER BY AantalTreasureHunts DESC
"""

result1_season = spark.sql(query1_season)
print("Seizoenen analyse:")
result1_season.show()

# Dagen van de week analyse
query1_days = """
SELECT
    d.DayOfTheWeek as DagVanDeWeek,
    CASE d.DayOfTheWeek
        WHEN 1 THEN 'Maandag'
        WHEN 2 THEN 'Dinsdag'
        WHEN 3 THEN 'Woensdag'
        WHEN 4 THEN 'Donderdag'
        WHEN 5 THEN 'Vrijdag'
        WHEN 6 THEN 'Zaterdag'
        WHEN 7 THEN 'Zondag'
    END as DagNaam,
    COUNT(*) as AantalTreasureHunts,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM factTreasureFound), 2) as Percentage
FROM factTreasureFound f
JOIN dimDate d ON f.DateSurKey = d.DateSurKey
GROUP BY d.DayOfTheWeek
ORDER BY d.DayOfTheWeek
"""

result1_days = spark.sql(query1_days)
print("Dagen van de week analyse:")
result1_days.show()

=== VRAAG 1: Datumparameters effect op Treasurehunts ===
Jaar/Maand/Week analyse:
+----+---------+----+-------------------+-----------------+----------------+
|Year|    Month|Week|AantalTreasureHunts|GemiddeldePerJaar|PercentageTotaal|
+----+---------+----+-------------------+-----------------+----------------+
|2020| December|  49|              10079|          9180.45|            0.56|
|2020| December|  50|              12023|          9180.45|            0.67|
|2020| December|  51|              11998|          9180.45|            0.67|
|2020| December|  52|              11260|          9180.45|            0.63|
|2020| December|  53|               6814|          9180.45|            0.38|
|2020| November|  44|               1627|          9180.45|            0.09|
|2020| November|  45|              11298|          9180.45|            0.63|
|2020| November|  46|              11460|          9180.45|            0.64|
|2020| November|  47|              11284|          9180.45|            

In [9]:
print("=== VRAAG 2: Regen effect op moeilijke terreinen ===")

query2 = """
SELECT
    r.RainDescription as RegenSituatie,
    CASE
        WHEN t.terrain >= 4.0 THEN 'Zeer Moeilijk'
        WHEN t.terrain >= 3.0 THEN 'Moeilijk'
        WHEN t.terrain >= 2.0 THEN 'Gemiddeld'
        ELSE 'Makkelijk'
    END as Terreinniveau,
    COUNT(*) as AantalCaches,
    ROUND(AVG(f.duration), 2) as GemiddeldeZoekTijd,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY r.RainDescription), 2) as PercentagePerRegen
FROM factTreasureFound f
JOIN dimRain r ON f.RainSurKey = r.RainSurKey
JOIN dimTreasureType t ON f.TreasureTypeSurKey = t.TreasureTypeSurKey
WHERE t.terrain IS NOT NULL
GROUP BY r.RainDescription,
    CASE
        WHEN t.terrain >= 4.0 THEN 'Zeer Moeilijk'
        WHEN t.terrain >= 3.0 THEN 'Moeilijk'
        WHEN t.terrain >= 2.0 THEN 'Gemiddeld'
        ELSE 'Makkelijk'
    END
ORDER BY r.RainDescription, Terreinniveau
"""

result2 = spark.sql(query2)
print("Regen effect op terrein moeilijkheid:")
result2.show()

# Gedetailleerde terrain analyse per regensituatie
query2_detailed = """
SELECT
    r.RainDescription as RegenSituatie,
    ROUND(t.terrain, 1) as TerreinScore,
    COUNT(*) as AantalCaches,
    ROUND(AVG(f.duration), 2) as GemiddeldeDuur,
    ROUND(MIN(f.duration), 2) as MinDuur,
    ROUND(MAX(f.duration), 2) as MaxDuur
FROM factTreasureFound f
JOIN dimRain r ON f.RainSurKey = r.RainSurKey
JOIN dimTreasureType t ON f.TreasureTypeSurKey = t.TreasureTypeSurKey
WHERE t.terrain IS NOT NULL
GROUP BY r.RainDescription, ROUND(t.terrain, 1)
ORDER BY r.RainDescription, TerreinScore
"""

result2_detailed = spark.sql(query2_detailed)
print("Gedetailleerde terrain analyse:")
result2_detailed.show(20)

=== VRAAG 2: Regen effect op moeilijke terreinen ===
Regen effect op terrein moeilijkheid:
+--------------------+-------------+------------+------------------+------------------+
|       RegenSituatie|Terreinniveau|AantalCaches|GemiddeldeZoekTijd|PercentagePerRegen|
+--------------------+-------------+------------+------------------+------------------+
|Regen situatie on...|    Gemiddeld|      730151|           4180.91|             40.80|
|Regen situatie on...|    Makkelijk|      484582|           4202.28|             27.08|
|Regen situatie on...|     Moeilijk|      330674|           4231.72|             18.48|
|Regen situatie on...|Zeer Moeilijk|      243975|           4224.66|             13.63|
|Weer met regen (c...|    Gemiddeld|           6|            5340.0|             20.00|
|Weer met regen (c...|    Makkelijk|           9|           4473.33|             30.00|
|Weer met regen (c...|     Moeilijk|          15|            1924.0|             50.00|
|   Weer zonder regen|    Gem

In [55]:
print("=== VRAAG 3: Weekend vs weekdag voor moeilijke caches ===")


# Combinatie van terrain EN difficulty
query3_combined = """
SELECT
    CASE
        WHEN d.IsWeekDay THEN 'Weekdag'
        ELSE 'Weekend'
    END as DagType,
    CASE
        WHEN t.difficulty >= 3.5 AND t.terrain >= 3.5 THEN 'Zeer Uitdagend'
        WHEN t.difficulty >= 2.5 OR t.terrain >= 2.5 THEN 'Gemiddeld'
        ELSE 'Makkelijk'
    END as Uitdaging,
    COUNT(*) as AantalCaches,
    ROUND(AVG(f.duration), 2) as GemiddeldeDuur
FROM factTreasureFound f
JOIN dimDate d ON f.DateSurKey = d.DateSurKey
JOIN dimTreasureType t ON f.TreasureTypeSurKey = t.TreasureTypeSurKey
WHERE t.difficulty IS NOT NULL AND t.terrain IS NOT NULL
GROUP BY
    CASE WHEN d.IsWeekDay THEN 'Weekdag' ELSE 'Weekend' END,
    CASE
        WHEN t.difficulty >= 3.5 AND t.terrain >= 3.5 THEN 'Zeer Uitdagend'
        WHEN t.difficulty >= 2.5 OR t.terrain >= 2.5 THEN 'Gemiddeld'
        ELSE 'Makkelijk'
    END
ORDER BY DagType, Uitdaging
"""

result3_combined = spark.sql(query3_combined)
print("Gecombineerde difficulty/terrain analyse:")
result3_combined.show()

=== VRAAG 3: Weekend vs weekdag voor moeilijke caches ===
Gecombineerde difficulty/terrain analyse:
+-------+--------------+------------+--------------+
|DagType|     Uitdaging|AantalCaches|GemiddeldeDuur|
+-------+--------------+------------+--------------+
|Weekdag|     Gemiddeld|      665113|       4217.88|
|Weekdag|     Makkelijk|      589093|        4182.6|
|Weekdag|Zeer Uitdagend|       23408|       4262.68|
|Weekend|     Gemiddeld|      265884|        4216.7|
|Weekend|     Makkelijk|      236526|       4180.72|
|Weekend|Zeer Uitdagend|        9414|       4263.46|
+-------+--------------+------------+--------------+



In [48]:
print("=== BIJKOMENDE VRAAG: In welke seizoenen worden de meeste caches gevonden? ===")

query_bonus1 = """
SELECT
    s.SeasonName as Seizoen,
    COUNT(*) as AantalCaches,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM factTreasureFound), 1) as Percentage
FROM factTreasureFound f
JOIN dimSeason s ON f.SeasonSurKey = s.SeasonSurKey
GROUP BY s.SeasonName
ORDER BY AantalCaches DESC
"""

result_bonus3 = spark.sql(query_bonus1)
result_bonus3.show()

=== BIJKOMENDE VRAAG: In welke seizoenen worden de meeste caches gevonden? ===
+-------+------------+----------+
|Seizoen|AantalCaches|Percentage|
+-------+------------+----------+
|  Zomer|      450073|      25.2|
|  Lente|      448330|      25.1|
| Herfst|      446989|      25.0|
| Winter|      444046|      24.8|
+-------+------------+----------+



In [49]:
print("=== BIJKOMENDE VRAAG: Worden er meer caches gevonden in het weekend? ===")

query_bonus2 = """
SELECT
    CASE
        WHEN d.IsWeekDay THEN 'Weekdag'
        ELSE 'Weekend'
    END as DagType,
    COUNT(*) as AantalCaches,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM factTreasureFound), 1) as Percentage,
    ROUND(AVG(f.duration), 1) as GemiddeldeZoekTijd
FROM factTreasureFound f
JOIN dimDate d ON f.DateSurKey = d.DateSurKey
GROUP BY
    CASE
        WHEN d.IsWeekDay THEN 'Weekdag'
        ELSE 'Weekend'
    END
ORDER BY AantalCaches DESC
"""

result_bonus4 = spark.sql(query_bonus2)
result_bonus4.show()

=== BIJKOMENDE VRAAG: Worden er meer caches gevonden in het weekend? ===
+-------+------------+----------+------------------+
|DagType|AantalCaches|Percentage|GemiddeldeZoekTijd|
+-------+------------+----------+------------------+
|Weekdag|     1277614|      71.4|            4202.4|
|Weekend|      511824|      28.6|            4200.9|
+-------+------------+----------+------------------+

